# Phase 1: Data Engineering & Banking Domain Feature Engineering

## Executive Context & Objectives
Customer churn in retail and commercial banking directly drives deposit flight and erosion of **Net Interest Margin (NIM)**.
In this notebook, we:
1. **Ingest & Clean** the 10,000-customer banking churn dataset.
2. **Validate Data Schemas** with strict constraints using **Pandera**.
3. **Engineer Banking Domain Features**:
   - **Wealth & Liquidity Metrics**: `BalanceToSalaryRatio`, `IsZeroBalance`, `WealthTier` (Zero Balance, Mass Market, Affluent, High Net Worth).
   - **Customer Stickiness & Life Cycle**: `TenureToAgeRatio`, `CreditScoreToAgeRatio`.
   - **Product Penetration & Complexity Risk**: `IsMultiProductRisk` ($NumOfProducts \ge 3$).
   - **Friction & Dissatisfaction Multipliers**: `ComplaintInactivityRisk`, `ComplaintRisk`.
   - **Loyalty & Rewards Index**: `LoyaltyIndex` (Tier weight $\times$ Satisfaction $\times$ Points).
4. **Analyze Distributions & Churn Drivers** with high-resolution visual analytics.
5. **Persist Leak-Free Preprocessing Pipelines** for downstream production modeling.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_PATH, PROCESSED_DATA_PATH, TARGET_COL
from src.preprocess import load_and_clean_data, BankingFeatureEngineer, build_preprocessor_pipeline
from src.data_schema import raw_bank_data_schema, engineered_bank_data_schema

pd.set_option('display.max_columns', 30)
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Raw Data Ingestion & Pandera Schema Validation

In [2]:
# Load and clean raw dataset (dropping non-predictive CustomerId, RowNumber, Surname)
df_clean = load_and_clean_data(validate=True)
print(f"Raw dataset shape after cleaning: {df_clean.shape}")
print(f"Target distribution:\n{df_clean[TARGET_COL].value_counts(normalize=True).round(4) * 100}%")
df_clean.head()

Raw dataset shape after cleaning: (10000, 15)
Target distribution:
Exited
0    79.62
1    20.38
Name: proportion, dtype: float64%


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,SatisfactionScore,CardType,PointEarned
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


## 2. Banking Domain Feature Engineering

In [3]:
fe = BankingFeatureEngineer()
df_engineered = fe.transform(df_clean)

# Validate engineered dataset against Pandera schema
df_validated = engineered_bank_data_schema.validate(df_engineered)
print(f"Engineered feature count: {df_engineered.shape[1]}")
print("Engineered schema validation passed!")
df_engineered[["Balance", "EstimatedSalary", "BalanceToSalaryRatio", "WealthTier", "TenureToAgeRatio", "IsMultiProductRisk", "ComplaintInactivityRisk", "LoyaltyIndex"]].head()

Engineered feature count: 24
Engineered schema validation passed!


,Balance,EstimatedSalary,BalanceToSalaryRatio,WealthTier,TenureToAgeRatio,IsMultiProductRisk,ComplaintInactivityRisk,LoyaltyIndex
0,0.00,101348.88,0.000000,Zero_Balance,0.047619,0,0,0.7424
1,83807.86,112542.58,0.744670,Affluent,0.024390,0,0,1.0944
2,159660.80,113931.57,1.401362,High_Net_Worth,0.190476,1,1,0.9048
3,0.00,93826.63,0.000000,Zero_Balance,0.025641,0,0,0.7000
4,125510.82,79084.10,1.587035,High_Net_Worth,0.046512,0,0,0.8500


## 3. Key Churn Dynamics & Statistical Insights

### Finding 1: The Service Complaint Multiplier
Customers who registered a complaint exhibit an extraordinary churn rate (>99% in this dataset), demonstrating that unresolved customer friction is the single strongest precursor to deposit flight.

In [4]:
complain_summary = df_engineered.groupby("Complain").agg(
    Total_Customers=("Exited", "count"),
    Churn_Rate=("Exited", "mean"),
    Avg_Balance=("Balance", "mean"),
    Total_Deposits=("Balance", "sum")
).reset_index()
complain_summary

,Complain,Total_Customers,Churn_Rate,Avg_Balance,Total_Deposits
0,0,7956,0.000503,72718.520926,5.785486e+08
1,1,2044,0.995108,91149.872989,1.863103e+08


### Finding 2: Product Bundling Paradox
Customers with 2 products exhibit the lowest churn (~7.6%), representing the banking sweet spot. Customers with 3 or 4 products have churn rates exceeding 80%, indicating product fatigue, mis-selling, or disjointed cross-selling.

In [5]:
prod_summary = df_engineered.groupby("NumOfProducts").agg(
    Customer_Count=("Exited", "count"),
    Churn_Rate=("Exited", "mean"),
    Avg_Loyalty_Points=("PointEarned", "mean"),
    Total_Deposits_at_Risk=("Balance", lambda x: x[df_engineered.loc[x.index, "Exited"] == 1].sum())
).reset_index()
prod_summary

,NumOfProducts,Customer_Count,Churn_Rate,Avg_Loyalty_Points,Total_Deposits_at_Risk
0,1,5084,0.277144,609.578088,1.296686e+08
1,2,4590,0.076035,603.696296,3.150084e+07
2,3,266,0.827068,603.894737,1.888768e+07
3,4,60,1.000000,574.233333,5.623988e+06


### Finding 3: Wealth Tiers & Deposit Flight Concentration
Affluent and High Net Worth customers represent over 70% of all balances at risk of departure, requiring targeted Relationship Manager retention workflows.

In [6]:
tier_summary = df_engineered.groupby("WealthTier").agg(
    Customer_Count=("Exited", "count"),
    Churn_Rate=("Exited", "mean"),
    Avg_Balance=("Balance", "mean"),
    Total_Deposits=("Balance", "sum"),
    Deposits_at_Risk=("Balance", lambda x: x[df_engineered.loc[x.index, "Exited"] == 1].sum())
).reset_index()
tier_summary

,WealthTier,Customer_Count,Churn_Rate,Avg_Balance,Total_Deposits,Deposits_at_Risk
0,Affluent,3127,0.238887,97520.152462,3.049455e+08,7.462528e+07
1,High_Net_Worth,3181,0.240490,143651.308152,4.569548e+08,1.100840e+08
2,Mass_Market,75,0.346667,39447.532000,2.958565e+06,9.718403e+05
3,Zero_Balance,3617,0.138236,0.000000,0.000000e+00,0.000000e+00


## 4. Leak-Free Preprocessor Pipeline Construction

In [7]:
from sklearn.model_selection import train_test_split
from src.preprocess import split_features_and_target

X, y = split_features_and_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = build_preprocessor_pipeline(X_train)
X_train_fe = fe.transform(X_train)
X_train_processed = preprocessor.fit_transform(X_train_fe)

print(f"X_train raw shape: {X_train.shape}")
print(f"X_train processed matrix shape: {X_train_processed.shape}")
print("Feature names out:", preprocessor.get_feature_names_out())

X_train raw shape: (8000, 14)
X_train processed matrix shape: (8000, 28)
Feature names out: ['num__CreditScore' 'num__Age' 'num__Tenure' 'num__Balance'
 'num__NumOfProducts' 'num__HasCrCard' 'num__IsActiveMember'
 'num__EstimatedSalary' 'num__Complain' 'num__SatisfactionScore'
 'num__PointEarned' 'num__BalanceToSalaryRatio' 'num__IsZeroBalance'
 'num__TenureToAgeRatio' 'num__CreditScoreToAgeRatio'
 'num__IsMultiProductRisk' 'num__ComplaintInactivityRisk'
 'num__ComplaintRisk' 'num__LoyaltyIndex' 'cat__Geography_Germany'
 'cat__Geography_Spain' 'cat__Gender_Male' 'cat__CardType_GOLD'
 'cat__CardType_PLATINUM' 'cat__CardType_SILVER'
 'cat__WealthTier_High_Net_Worth' 'cat__WealthTier_Mass_Market'
 'cat__WealthTier_Zero_Balance']
